In [1]:
import pandas as pd

df = pd.read_csv("heart.csv")
df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [2]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

In [3]:
df.rename(columns={
    "trestbps": "bp",
    "chol": "cholesterol"
}, inplace=True)

In [4]:
df["target"].value_counts()

target
1    526
0    499
Name: count, dtype: int64

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
# from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Define features
categorical = ["cp", "restecg", "slope", "thal"]
numerical = ["age", "sex", "bp", "cholesterol", "fbs", "thalach", "exang", "oldpeak", "ca"]

# Preprocessing
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), categorical),
    ("num", StandardScaler(), numerical)
])

# Model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        class_weight="balanced",
        random_state=42
    ))
])

# Prepare data
X = df.drop("target", axis=1)
y = df["target"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
model.fit(X_train, y_train)

# Evaluate
from sklearn.metrics import accuracy_score
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 1.0


In [6]:
import sklearn
print(sklearn.__version__)

1.9.0


In [7]:
import joblib
joblib.dump(model, "../backend/models/heart_model.pkl")

['../backend/models/heart_model.pkl']

In [8]:
import pandas as pd

# Load data
df = pd.read_csv("heart.csv")

# Rename columns (IMPORTANT)
df.rename(columns={
    "trestbps": "bp",
    "chol": "cholesterol"
}, inplace=True)

# ✅ FIX: Treat categorical as categorical (NOT numbers)
categorical = ["cp", "restecg", "slope", "thal"]
for col in categorical:
    df[col] = df[col].astype(str)

# Feature split
X = df.drop("target", axis=1)
y = df["target"]

# Columns
numerical = ["age", "sex", "bp", "cholesterol", "fbs", "thalach", "exang", "oldpeak", "ca"]

# 🚀 Pipeline imports
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

# Preprocessing
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first"), categorical),
    ("num", StandardScaler(), numerical)
])

# ✅ SVM MODEL (TUNED)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", SVC(
        probability=True,
        class_weight="balanced",
        kernel="rbf",
        C=1.0,
        gamma="scale"
    ))
])

# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train
model.fit(X_train, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

# Cross validation (IMPORTANT)
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5)
print("Cross Validation:", scores.mean())


c:\Users\HP\OneDrive\Desktop\smart_healthcare_system\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy: 0.8634146341463415


c:\Users\HP\OneDrive\Desktop\smart_healthcare_system\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\OneDrive\Desktop\smart_healthcare_system\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\OneDrive\Desktop\smart_healthcare_system\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\HP\OneDrive\Desktop\smart_healthcare_system\venv\Lib\site-packages\

Cross Validation: 0.9160975609756097


In [9]:
test_input = pd.DataFrame([{
  "age": 45,
  "sex": 1,
  "cp": "3",
  "bp": 120,
  "cholesterol": 200,
  "fbs": 0,
  "restecg": "1",
  "thalach": 150,
  "exang": 0,
  "oldpeak": 0.5,
  "slope": "2",
  "ca": 0,
  "thal": "2"
}])

print(model.predict(test_input))
print(model.predict_proba(test_input))

[1]
[[0.01568515 0.98431485]]


In [10]:
# 💾 SAVE MODEL
import joblib
joblib.dump(model, "../backend/models/heart_model.pkl")

['../backend/models/heart_model.pkl']

In [11]:
# from sklearn.linear_model import LogisticRegression

# model = LogisticRegression(max_iter=1000)

# model.fit(X_train_scaled, y_train)

In [12]:
# from sklearn.metrics import accuracy_score, classification_report

# y_pred = model.predict(X_test_scaled)

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print(classification_report(y_test, y_pred))

In [13]:
# from sklearn.ensemble import RandomForestClassifier

# rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# rf_model.fit(X_train, y_train)

# rf_pred = rf_model.predict(X_test)

In [14]:
# from sklearn.model_selection import cross_val_score

# scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5)

# print("SVM Cross Validation Accuracy:", scores.mean())

| Step                 | Status |
| -------------------- | ------ |
| Cleaning             | ✅      |
| Encoding             | ✅      |
| Scaling              | ✅      |
| Multi-model training | ✅      |
| Model selection      | ✅      |
| Saving model         | ✅      |
